## Imports

In [40]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from argparse import Namespace
from sklearn.utils import compute_class_weight

## Loading Dataset

In [2]:
df = pd.read_csv('./../data/processed/20newsgroup_preprocessed_own.csv', on_bad_lines='skip', delimiter=";")
print(df.shape)
df.head()

(18828, 3)


,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,kings become philosophers philosophers become ...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


In [3]:
df = df.dropna(subset=['text_cleaned'])
print(df.shape)

(18792, 3)


## Prepare BERT

In [59]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # fast and decent quality
X_bert = model.encode(df['text_cleaned'].tolist(), show_progress_bar=True)
print(X_bert.shape)
print(X_bert[:2])

Batches:   0%|          | 0/588 [00:00<?, ?it/s]

(18792, 384)
[[ 1.25527224e-02 -4.10585329e-02 -5.11534400e-02  6.39102096e-03
   8.84484425e-02  1.01927649e-02 -8.46677646e-02 -3.89941745e-02
   3.25094946e-02  4.10558879e-02 -5.26617467e-03 -4.93914597e-02
   2.13528145e-03 -1.11915777e-02 -7.73477368e-03  5.99905849e-02
  -1.17425434e-02  1.96864363e-02  5.23262471e-03 -1.71133224e-02
  -3.48223597e-02  1.04444586e-01 -2.15555425e-03  2.18533562e-03
  -2.21658666e-02 -3.73452045e-02 -9.62073635e-03 -4.25627455e-02
  -4.88256626e-02 -1.35162249e-02 -5.01156226e-02  3.01868469e-03
   3.81938145e-02 -2.59652436e-02  4.35806066e-02 -1.74001846e-02
   6.28648922e-02  8.70951265e-02  9.21683982e-02 -6.13419898e-02
  -4.53861766e-02 -3.55406851e-02 -7.29432777e-02 -5.45754135e-02
  -3.55321504e-02 -6.32485375e-03 -7.66699910e-02 -3.69550921e-02
   5.96298045e-03 -6.72317520e-02 -8.38710368e-02 -3.15112397e-02
   2.60465965e-02  4.68660109e-02 -9.24395472e-02  2.49064621e-02
  -8.65275785e-03 -1.50543144e-02 -4.69682068e-02 -1.07651465e-

## Encode target labels

In [52]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['target'])
print("Unique labels in df:", np.unique(df['target']))
print("Unique labels in full dataset:", np.unique(y_encoded))

Unique labels in df: ['alt.atheism' 'comp.graphics' 'comp.os.ms-windows.misc'
 'comp.sys.ibm.pc.hardware' 'comp.sys.mac.hardware' 'comp.windows.x'
 'misc.forsale' 'rec.autos' 'rec.motorcycles' 'rec.sport.baseball'
 'rec.sport.hockey' 'sci.crypt' 'sci.electronics' 'sci.med' 'sci.space'
 'soc.religion.christian' 'talk.politics.guns' 'talk.politics.mideast'
 'talk.politics.misc' 'talk.religion.misc']
Unique labels in full dataset: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


## Create dataset and dataloader

In [61]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y_encoded, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_bert, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

## Create a simple NN

In [65]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.main = nn.Sequential(
            nn.Linear(cfg.n_in, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.n_hidden, cfg.n_out)  # raw logits for CrossEntropyLoss
        )

    def forward(self, x):
        return self.main(x)

## Model training

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Namespace(
    n_in = X_bert.shape[1], 
    n_hidden = 256, 
    n_out = 20

)
model = MultilayerPerceptron(cfg).to(device)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print(class_weights)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

[1.17629108 0.96862113 0.95873724 0.95996169 0.98126632 0.96118926
 0.97616883 0.9514557  0.94666247 0.94666247 0.94191729 0.94785624
 0.95751592 0.9514557  0.9526616  0.94191729 1.03248626 0.99953457
 1.21429725 1.494334  ]
Epoch 1, Loss: 2.7774
Epoch 2, Loss: 2.7682
Epoch 3, Loss: 2.7661
Epoch 4, Loss: 2.7656
Epoch 5, Loss: 2.7647


## Evaluation

In [56]:
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_labels = []
labels = list(range(20))

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1).cpu()
        all_preds.extend(preds)
        all_labels.extend(y_batch)

print(classification_report(all_labels, all_preds, labels=labels, zero_division=0))

              precision    recall  f1-score   support

           0       0.22      0.04      0.06       798
           1       0.28      0.02      0.03       970
           2       0.00      0.00      0.00       980
           3       0.24      0.35      0.29       979
           4       0.01      0.09      0.02        32
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         0
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         0
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         0
          11       0.00      0.00      0.00         0
          12       0.00      0.00      0.00         0
          13       0.00      0.00      0.00         0
          14       0.00      0.00      0.00         0
          15       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0
          17       0.00    